# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
TASK_ID='task175'
VERIFY_SCOPE='visible'  # change to 'all' for train+test+arc-gen if available
MODEL_VERSION='task175-symbolic-v2-public-30x30'

In [2]:
# ONNX dependency setup.
import importlib.util, subprocess, sys
required = {'onnx':'onnx', 'onnxruntime':'onnxruntime', 'sklearn':'scikit-learn', 'torch':'torch'}
missing = [pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import onnx, onnxruntime as ort
print('onnx:', onnx.__version__)
print('onnxruntime:', ort.__version__)

Installing missing packages: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 68.6 MB/s eta 0:00:00
onnx: 1.20.1
onnxruntime: 1.27.0


In [3]:
import json, os, zipfile, subprocess, sys, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort
CH=10
PUBLIC_H=30
PUBLIC_W=30
FORBIDDEN={'Loop','Scan','NonZero','Unique','Script','Function'}

def find_task_json(task_id):
    candidates=[Path.cwd()/f'{task_id}.json',Path('/mnt/data')/f'{task_id}.json',Path('/kaggle/working')/f'{task_id}.json']
    for p in candidates:
        if p.exists(): return p
    for base in [Path('/kaggle/input'),Path.cwd(),Path('/kaggle/working')]:
        if base.exists():
            hits=list(base.rglob(f'{task_id}.json'))
            if hits: return hits[0]
    raise FileNotFoundError(task_id)

def load_task(task_id):
    p=find_task_json(task_id)
    with open(p) as f: return json.load(f),p

def examples_for_scope(task,scope='visible'):
    if scope=='all': return task.get('train',[])+task.get('test',[])+task.get('arc-gen',[])
    return task.get('train',[])+task.get('test',[])

def grid_to_tensor30(grid):
    arr=np.zeros((1,CH,PUBLIC_H,PUBLIC_W),np.float32)
    for r,row in enumerate(grid[:PUBLIC_H]):
        for c,v in enumerate(row[:PUBLIC_W]):
            arr[0,int(v),r,c]=1.0
    return arr

def validate_model(model_path,task,scope='visible'):
    sess=ort.InferenceSession(str(model_path),providers=['CPUExecutionProvider'])
    right=0; total=0; first_wrong=None; first_wrong_pixels=None
    for i,ex in enumerate(examples_for_scope(task,scope)):
        pred=(sess.run(['output'],{'input':grid_to_tensor30(ex['input'])})[0]>0.5).astype(np.float32)
        exp=grid_to_tensor30(ex['output'])
        total+=1
        if np.array_equal(pred,exp): right+=1
        elif first_wrong is None:
            first_wrong=i; first_wrong_pixels=int(np.sum(pred!=exp))
    m=onnx.load(str(model_path)); ops={}
    for n in m.graph.node: ops[n.op_type]=ops.get(n.op_type,0)+1
    input_shape=[d.dim_value if d.HasField('dim_value') else d.dim_param for d in m.graph.input[0].type.tensor_type.shape.dim]
    output_shape=[d.dim_value if d.HasField('dim_value') else d.dim_param for d in m.graph.output[0].type.tensor_type.shape.dim]
    return {'right':right,'total':total,'first_wrong':first_wrong,'first_wrong_pixels':first_wrong_pixels,
            'file_size_bytes':Path(model_path).stat().st_size,'under_1_4mb':Path(model_path).stat().st_size<1_400_000,
            'forbidden_ops_present':sorted(FORBIDDEN & set(ops)),'op_counts':ops,
            'input_shape':input_shape,'output_shape':output_shape}


def set_static_io_shapes(model_path):
    m=onnx.load(str(model_path))
    for vi in [m.graph.input[0], m.graph.output[0]]:
        for dim,value in zip(vi.type.tensor_type.shape.dim,[1,10,30,30]):
            dim.ClearField('dim_param')
            dim.dim_value=value
    onnx.checker.check_model(m)
    onnx.save(m,str(model_path))

class Public30Wrapper(nn.Module):
    def __init__(self, core, canvas_h, canvas_w):
        super().__init__()
        self.core=core
        self.canvas_h=canvas_h
        self.canvas_w=canvas_w
    def forward(self,x):
        y=self.core(x[:,:,:self.canvas_h,:self.canvas_w])
        return F.pad(y,(0,PUBLIC_W-self.canvas_w,0,PUBLIC_H-self.canvas_h),mode='constant',value=0.0)

In [4]:
class Task175EuclideanQuotient(nn.Module):
    def __init__(self,H,W):
        super().__init__()
        R,C=np.indices((H,W))
        mn=np.minimum(R,C); mx=np.maximum(R,C); diff=mx-mn; q=diff//(mn+2)
        self.register_buffer('q',torch.from_numpy(q.astype(np.float32)).view(1,1,H,W))
        self.register_buffer('diff0',torch.from_numpy((diff==0).astype(np.float32)).view(1,1,H,W))
        self.register_buffer('diag_mask',torch.eye(H,W).view(1,1,H,W))
        self.register_buffer('color_values',torch.arange(10,dtype=torch.float32).view(1,10,1,1))
        self.register_buffer('vals10',torch.arange(10,dtype=torch.float32).view(1,10,1,1))
    def forward(self,x):
        vals=(x*self.color_values).sum(dim=1,keepdim=True)
        K=torch.amax(vals,dim=(2,3),keepdim=True)
        D=torch.amax(vals*self.diag_mask,dim=(2,3),keepdim=True)
        raw=torch.remainder(D+self.q-2.0,K)+1.0
        raw=raw*(1.0-self.diff0)+D*self.diff0
        return (torch.abs(raw-self.vals10)<0.5).float()

In [5]:
task,task_path=load_task(TASK_ID)
CANVAS_H=21; CANVAS_W=21
OUT_DIR=Path.cwd()/f'working_submission_{TASK_ID}'
OUT_DIR.mkdir(parents=True,exist_ok=True)
MODEL_PATH=OUT_DIR/f'{TASK_ID}.onnx'
print('task path:',task_path)
print('train:',len(task.get('train',[])),'test:',len(task.get('test',[])),'arc-gen:',len(task.get('arc-gen',[])))
print('native canvas:',CANVAS_H,CANVAS_W)
print('public ONNX shape:',(1,CH,PUBLIC_H,PUBLIC_W))
print('MODEL_PATH:',MODEL_PATH)

task path: /kaggle/input/competitions/neurogolf-2026/task175.json
train: 3 test: 1 arc-gen: 262
native canvas: 21 21
public ONNX shape: (1, 10, 30, 30)
MODEL_PATH: /kaggle/working/working_submission_task175/task175.onnx


In [6]:
# Build ONNX model and enforce competition constraints.
core=Task175EuclideanQuotient(CANVAS_H,CANVAS_W)
model=Public30Wrapper(core,CANVAS_H,CANVAS_W)
model.eval()
torch.onnx.export(model, torch.zeros(1,CH,PUBLIC_H,PUBLIC_W,dtype=torch.float32), str(MODEL_PATH), input_names=['input'], output_names=['output'], opset_version=17, dynamic_axes=None, do_constant_folding=True, dynamo=False)
onnx_model=onnx.load(str(MODEL_PATH)); onnx_model.ir_version=8; onnx.checker.check_model(onnx_model); onnx.save(onnx_model,str(MODEL_PATH))
set_static_io_shapes(MODEL_PATH)
validation_report=validate_model(MODEL_PATH,task,VERIFY_SCOPE)
assert validation_report['right']==validation_report['total'], validation_report
assert validation_report['under_1_4mb'], validation_report['file_size_bytes']
assert not validation_report['forbidden_ops_present'], validation_report['forbidden_ops_present']
assert validation_report['input_shape']==[1,10,30,30], validation_report['input_shape']
assert validation_report['output_shape']==[1,10,30,30], validation_report['output_shape']
validation_report

/tmp/ipykernel_16/3507245149.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, torch.zeros(1,CH,PUBLIC_H,PUBLIC_W,dtype=torch.float32), str(MODEL_PATH), input_names=['input'], output_names=['output'], opset_version=17, dynamic_axes=None, do_constant_folding=True, dynamo=False)
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/jit_utils.py:305: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /pytorch/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inferenc

{'right': 4,
 'total': 4,
 'first_wrong': None,
 'first_wrong_pixels': None,
 'file_size_bytes': 9978,
 'under_1_4mb': True,
 'forbidden_ops_present': [],
 'op_counts': {'Identity': 2,
  'Constant': 21,
  'Slice': 3,
  'Mul': 5,
  'ReduceSum': 1,
  'ReduceMax': 2,
  'Add': 3,
  'Sub': 3,
  'Div': 1,
  'Floor': 1,
  'Abs': 1,
  'Less': 1,
  'Cast': 2,
  'ConstantOfShape': 1,
  'Concat': 1,
  'Reshape': 2,
  'Transpose': 1,
  'Pad': 1},
 'input_shape': [1, 10, 30, 30],
 'output_shape': [1, 10, 30, 30]}

In [7]:
visible_report=validate_model(MODEL_PATH,task,'visible')
print('visible_report:',visible_report)
if task.get('arc-gen'):
    all_report=validate_model(MODEL_PATH,task,'all')
    print('all_report:',all_report)
else:
    all_report=None

visible_report: {'right': 4, 'total': 4, 'first_wrong': None, 'first_wrong_pixels': None, 'file_size_bytes': 9978, 'under_1_4mb': True, 'forbidden_ops_present': [], 'op_counts': {'Identity': 2, 'Constant': 21, 'Slice': 3, 'Mul': 5, 'ReduceSum': 1, 'ReduceMax': 2, 'Add': 3, 'Sub': 3, 'Div': 1, 'Floor': 1, 'Abs': 1, 'Less': 1, 'Cast': 2, 'ConstantOfShape': 1, 'Concat': 1, 'Reshape': 2, 'Transpose': 1, 'Pad': 1}, 'input_shape': [1, 10, 30, 30], 'output_shape': [1, 10, 30, 30]}
all_report: {'right': 266, 'total': 266, 'first_wrong': None, 'first_wrong_pixels': None, 'file_size_bytes': 9978, 'under_1_4mb': True, 'forbidden_ops_present': [], 'op_counts': {'Identity': 2, 'Constant': 21, 'Slice': 3, 'Mul': 5, 'ReduceSum': 1, 'ReduceMax': 2, 'Add': 3, 'Sub': 3, 'Div': 1, 'Floor': 1, 'Abs': 1, 'Less': 1, 'Cast': 2, 'ConstantOfShape': 1, 'Concat': 1, 'Reshape': 2, 'Transpose': 1, 'Pad': 1}, 'input_shape': [1, 10, 30, 30], 'output_shape': [1, 10, 30, 30]}


In [8]:
manifest={'task_id':TASK_ID,'model_version':MODEL_VERSION,'rule':'adaptive 2D residue-period repair','validation_report':validation_report,'visible_report':visible_report}
manifest_path=OUT_DIR/f'{TASK_ID}_manifest.json'
with open(manifest_path,'w') as f: json.dump(manifest,f,indent=2)
zip_path=OUT_DIR/f'{TASK_ID}_submission.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf: zf.write(MODEL_PATH,MODEL_PATH.name)
print('wrote',MODEL_PATH)
print('wrote',manifest_path)
print('wrote',zip_path)

submission_path=Path.cwd()/'submission.zip'
with zipfile.ZipFile(submission_path,'w',zipfile.ZIP_DEFLATED) as zf: zf.write(MODEL_PATH, MODEL_PATH.name)

wrote /kaggle/working/working_submission_task175/task175.onnx
wrote /kaggle/working/working_submission_task175/task175_manifest.json
wrote /kaggle/working/working_submission_task175/task175_submission.zip
